# Notebook 37 — Reproducing the RMM index (Wheeler & Hendon 2004) from its original data

Rebuilds RMM1/RMM2 with the paper's data (NOAA interpolated OLR + NCEP-NCAR R1 u850/u200, 2.5°), domain
(15S-15N mean, all longitudes), EOF period (1979-2001, 8401 days) and method. The result is checked against the
paper's published numbers and against the official BoM index. Inputs come from nb36.

**Deliberate deviation (user decision, Session 65): the SST1 regression step is skipped.** WH04 removes the part
of each field that is linearly related to SST1 (a rotated EOF of Indo-Pacific SST; Drosdowsky & Chambers 2001)
before subtracting the 120-day mean. That series is not available here. According to the BoM file header, the
official RMM *does* include this step from 1974 to 2013. From 2014 BoM uses Gottschalck et al. (2010): 120-day
mean only, with ACCESS winds. The reproduction is therefore expected to come close to the official index in
1979-2001 but not match it exactly. The difference is the SST1 component plus any change in dataset versions.

**Details the paper leaves open, decided by a small grid (Cell 4):**
- whether the 120-day window includes day t;
- how the 3-harmonic annual cycle is fitted (to calendar-day means, or directly to the daily data);
- whether the 15S-15N mean is cos-weighted.

Each of the 8 combinations is scored by its correlation with the official RMM over 1979-2001. The best one is kept.

**Outputs:**
- `data/index_repro/rmm_wh04_repro.npz` — ladder row R2: WH04 method, no SST1, NOAA + R1.
- `data/index_repro/rmm_wh04_era5.npz` — ladder row R3: same method on the ERA5 fields (X_MJO), regridded to 2.5°, EOF on 1979-2001.
- `results/index_repro/rmm_*` — tables and figures.

## Cell 1 — Setup + published WH04 numbers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, itertools, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
IDX_DIR  = f'{PROJECT_DIR}/data/index_repro'
OUT_DIR  = f'{PROJECT_DIR}/results/index_repro'
MJO_PROC = f'{PROJECT_DIR}/MJO/data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

EOF_PERIOD = ('1979-01-01', '2001-12-31')   # WH04: 8401 days
N_HARM, WINDOW = 3, 120
USE_SST1 = False                            # Session 65: skipped (SST1 series not available)
FIELDS = ['olr', 'u850', 'u200']

WH04 = {                                    # targets read from the paper
    'n_days': 8401,
    'expl_var_pct': [12.8, 12.2, 6.1],
    'band30_80_frac': [0.60, 0.62, 0.31],    # Fig. 2 (PC1, PC2, PC3)
    'coh2_30_80': 0.76,                      # Fig. 3a
    'lagcorr_max': (0.56, 9),                # Fig. 6: RMM1 vs RMM2, RMM2 lagging
    'fig1_extremes': {'EOF1': {'olr': -8.5, 'u850': 1.2, 'u200': 2.7},
                      'EOF2': {'olr': 11.7, 'u850': 1.5, 'u200': 2.8}},   # approximate, read off Fig. 1
}

def wh04_phase(x, y):
    """WH04 Fig. 7 phases on the (x, y) plane: phase 1 = 180-225 deg, ..., phase 5 = 0-45 deg."""
    ang = np.degrees(np.arctan2(y, x)) % 360
    return (np.floor(((ang - 180) % 360) / 45).astype(int) % 8) + 1
print('USE_SST1 =', USE_SST1)

## Cell 2 — Load nb36 data and the official RMM

In [ ]:
o = np.load(f'{IDX_DIR}/noaa_olr_15S15N.npz', allow_pickle=True)
w = np.load(f'{IDX_DIR}/ncep_r1_uwnd_15S15N.npz', allow_pickle=True)
lat, lon = o['lat'].astype(float), o['lon'].astype(float)
assert np.allclose(lat, w['lat']) and np.allclose(lon, w['lon']), 'OLR and R1 grids differ'
d_o, d_w = pd.DatetimeIndex(o['dates']), pd.DatetimeIndex(w['dates'])
dates = pd.date_range(max(d_o[0], d_w[0]), min(d_o[-1], d_w[-1]), freq='D')

def on_days(arr, d):
    idx = d.get_indexer(dates)
    out = np.full((len(dates),) + arr.shape[1:], np.nan, np.float32)
    out[idx >= 0] = arr[idx[idx >= 0]]
    return out

RAW = {'olr': on_days(o['olr'], d_o), 'u850': on_days(w['u850'], d_w), 'u200': on_days(w['u200'], d_w)}
base = (dates >= EOF_PERIOD[0]) & (dates <= EOF_PERIOD[1])
print(f'days {dates[0].date()} .. {dates[-1].date()} ({len(dates)}); grid {len(lat)} lat x {len(lon)} lon; '
      f'EOF-period days {int(base.sum())}')

lab = pd.read_csv(f'{MJO_PROC}/labels_aligned_mjo.csv', parse_dates=['date'])
lab['date'] = lab['date'].dt.normalize()
lab = lab.set_index('date')
off = lab[['rmm1', 'rmm2']].reindex(dates).values.astype(float)
off[np.abs(off) > 100] = np.nan                                   # 1E36 / 999 missing codes
off_phase = lab['phase'].reindex(dates).values
off_amp = np.hypot(off[:, 0], off[:, 1])
_ok = np.isfinite(off).all(1) & (off_amp >= 1)
off_ph_calc = np.zeros(len(dates), int)
off_ph_calc[np.isfinite(off).all(1)] = wh04_phase(off[np.isfinite(off).all(1), 0], off[np.isfinite(off).all(1), 1])
print(f'BoM phase column vs WH04 phase computed from BoM RMM1/RMM2 (active days): '
      f'{(off_ph_calc[_ok] == off_phase[_ok]).mean():.3f} agreement (1.000 = convention confirmed)')
PERIODS = {'1979-2001 (EOF period; BoM includes SST1)': (dates >= '1979-01-01') & (dates <= '2001-12-31'),
           '2002-2013 (BoM: WH04 method, SST1)':        (dates >= '2002-01-01') & (dates <= '2013-12-31'),
           '2014-2022 (BoM: no SST1, ACCESS winds)':    (dates >= '2014-01-01') & (dates <= '2022-12-31')}
print('official RMM days available:', int(np.isfinite(off).all(1).sum()))

## Cell 3 — The WH04 pipeline

1. 15S-15N mean.
2. Remove the annual cycle: time mean + 3 harmonics, fitted over 1979-2001.
3. (SST1 step: skipped.)
4. Subtract the mean of the previous 120 days.
5. Divide each field by the square root of its global (all-longitude) variance over 1979-2001.
6. EOF of the combined 432-element vector (covariance matrix) over the valid days of 1979-2001.
7. Project every day onto the EOFs.
8. Divide each PC by its 1979-2001 standard deviation.

EOF signs follow WH04 Fig. 1:
- EOF1 has negative OLR over the Maritime Continent (100-140E);
- EOF2 has negative OLR over the western Pacific (150E-180).

In [ ]:
def merid_mean(F, how):
    if how == 'cos':
        wts = np.cos(np.deg2rad(lat)); wts = wts / wts.sum()
        return np.einsum('tyx,y->tx', F, wts)
    return F.mean(axis=1)

def harmonics(d, K=N_HARM):
    doy = d.dayofyear.values
    ylen = np.where(d.is_leap_year, 366.0, 365.0)
    ph = 2 * np.pi * (doy - 0.5) / ylen
    return np.column_stack([np.ones(len(d))] + [f(k * ph) for k in range(1, K + 1) for f in (np.cos, np.sin)])

def remove_annual_cycle(F, d, fit_mask, how):
    X = harmonics(d)
    ok = fit_mask & np.isfinite(F).all(axis=1)
    if how == 'direct':                                   # least squares on the daily data
        coef = np.linalg.lstsq(X[ok], F[ok], rcond=None)[0]
    else:                                                 # 'doyclim': fit the calendar-day climatology
        key = pd.Index(d[ok].strftime('%m-%d'))
        clim = pd.DataFrame(F[ok], index=key).groupby(level=0).mean()
        rep = pd.Series(np.arange(int(ok.sum())), index=key).groupby(level=0).first()
        coef = np.linalg.lstsq(X[ok][rep.loc[clim.index].values], clim.values, rcond=None)[0]
    return F - X @ coef

def remove_prev_mean(F, include_today):
    """'previous 120 days': t-120..t-1 (closed='left') or t-119..t (closed='right'); NaN days are skipped."""
    rm = pd.DataFrame(F).rolling(WINDOW, min_periods=1, closed='right' if include_today else 'left').mean().values
    return F - rm

def global_sd(F, mask):
    return float(np.sqrt(np.nanmean(np.nanvar(F[mask], axis=0))))

def eof(Z, mask, k=10):
    Zb = Z[mask]
    U, s, Vt = np.linalg.svd(Zb - Zb.mean(0), full_matrices=False)
    lam = s ** 2 / (len(Zb) - 1)
    return Vt[:k], lam[:k], float(lam.sum())

def sign_wh04(E, lon_):
    E = E.copy(); nl = len(lon_)
    mc, wp = (lon_ >= 100) & (lon_ <= 140), (lon_ >= 150) & (lon_ <= 180)
    if E[0, :nl][mc].mean() > 0: E[0] *= -1
    if E[1, :nl][wp].mean() > 0: E[1] *= -1
    return E

def finish(Z, fac, mask_fit, lon_, cfg):
    """EOF on mask_fit days -> signed EOFs, PCs (all days), normalised PCs."""
    valid = np.isfinite(Z).all(axis=1)
    fit = mask_fit & valid
    E, lam, tot = eof(Z, fit)
    E[:2] = sign_wh04(E[:2], lon_)
    pcs = np.full((len(Z), 3), np.nan)
    pcs[valid] = Z[valid] @ E[:3].T
    sd = pcs[fit].std(axis=0)
    return dict(cfg=cfg, E=E, lam=lam, frac=lam / tot, fac=fac, pcs=pcs, pcn=pcs / sd, sd=sd,
                valid=valid, fit=fit, n_fit=int(fit.sum()))

def build_rmm(cfg):
    anom, fac = {}, {}
    for f in FIELDS:
        F = merid_mean(RAW[f], cfg['merid'])
        F = remove_annual_cycle(F, dates, base, cfg['annual'])
        F = remove_prev_mean(F, cfg['window'] == 'incl_today')
        fac[f] = global_sd(F, base)
        anom[f] = F / fac[f]
    Z = np.concatenate([anom[f] for f in FIELDS], axis=1)          # [OLR | u850 | u200], 432 columns
    return finish(Z, fac, base, lon, cfg)

def canon(A, B):
    qa, _ = np.linalg.qr(A - A.mean(0)); qb, _ = np.linalg.qr(B - B.mean(0))
    return np.linalg.svd(qa.T @ qb, compute_uv=False)

def procrustes_deg(A, B):
    U, _, Vt = np.linalg.svd(A.T @ B); R = U @ Vt
    return float(np.degrees(np.arctan2(R[1, 0], R[0, 0]))), float(np.sign(np.linalg.det(R)))

def compare(pcn, mask):
    m = mask & np.isfinite(pcn[:, :2]).all(1) & np.isfinite(off).all(1)
    C = np.corrcoef(np.c_[pcn[m, :2], off[m]].T)[:2, 2:]
    A = (pcn[m, :2] - pcn[m, :2].mean(0)) / pcn[m, :2].std(0)
    B = (off[m] - off[m].mean(0)) / off[m].std(0)
    ang, det = procrustes_deg(A, B)
    act = m & (off_amp >= 1)
    ph = wh04_phase(pcn[act, 0], pcn[act, 1])
    agree = float((ph == off_ph_calc[act]).mean()) if act.any() else np.nan
    return {'n': int(m.sum()), 'r1': C[0, 0], 'r2': C[1, 1], 'r12': C[0, 1], 'r21': C[1, 0],
            'canon1': canon(A, B)[0], 'canon2': canon(A, B)[1], 'procrustes_deg': ang, 'det': det,
            'rmse1': float(np.sqrt(np.mean((pcn[m, 0] - off[m, 0]) ** 2))),
            'rmse2': float(np.sqrt(np.mean((pcn[m, 1] - off[m, 1]) ** 2))),
            'phase_agree_active': agree}
print('Pipeline ready.')

## Cell 4 — Grid over the details the paper leaves open (scored against the official RMM, 1979-2001)

In [ ]:
VARIANTS = [dict(window=w, annual=a, merid=m) for w, a, m in
            itertools.product(['excl_today', 'incl_today'], ['doyclim', 'direct'], ['mean', 'cos'])]
p0 = list(PERIODS)[0]
rows, runs = [], {}
for cfg in VARIANTS:
    t0 = time.time()
    r = build_rmm(cfg)
    c = compare(r['pcn'], PERIODS[p0])
    key = f"{cfg['window']}|{cfg['annual']}|{cfg['merid']}"
    runs[key] = r
    rows.append({'variant': key, 'n_fit_days': r['n_fit'], 'EOF1_%': 100 * r['frac'][0], 'EOF2_%': 100 * r['frac'][1],
                 'EOF3_%': 100 * r['frac'][2], 'r_RMM1': c['r1'], 'r_RMM2': c['r2'], 'canon_min': c['canon2'],
                 'phase_agree': c['phase_agree_active'], 'sec': time.time() - t0})
grid = pd.DataFrame(rows)
grid['score'] = grid[['r_RMM1', 'r_RMM2']].mean(axis=1)
grid = grid.sort_values('score', ascending=False).reset_index(drop=True)
grid.to_csv(f'{OUT_DIR}/rmm_variant_grid.csv', index=False)
pd.set_option('display.width', 200)
print(grid.round(4).to_string(index=False))
BEST = grid.loc[0, 'variant']
best = runs[BEST]
print(f'\nBest variant: {BEST}')

## Cell 5 — Validation of the best variant

The paper's numbers and the official index, side by side. A per-component r of at least 0.99 would mean an
exact reproduction. Without SST1, expect slightly less in the SST1 era (1979-2013).

In [ ]:
def band_frac(x, lo_days, hi_days):
    x = x - x.mean(); P = np.abs(np.fft.rfft(x)) ** 2; f = np.fft.rfftfreq(len(x))
    band = (f >= 1 / hi_days) & (f <= 1 / lo_days)
    return float(P[band].sum() / P[1:].sum())

def coh2_band(x, y, lo_days, hi_days, nperseg=512):
    f, C = signal.coherence(x, y, fs=1.0, nperseg=nperseg)
    band = (f >= 1 / hi_days) & (f <= 1 / lo_days)
    return float(C[band].mean())

def lagcorr_max(x, y, maxlag=25):
    out = [(np.corrcoef(x[:len(x) - L], y[L:])[0, 1], L) for L in range(0, maxlag + 1)]
    return max(out)

fit = best['fit']
P1, P2, P3 = (best['pcs'][fit, k] for k in range(3))
ext = {}
for k in range(2):
    ph = best['E'][k] * np.sqrt(best['lam'][k])                     # 1-SD PC anomaly, normalised units
    for i, f in enumerate(FIELDS):
        seg = ph[i * len(lon):(i + 1) * len(lon)] * best['fac'][f]
        ext[(f'EOF{k + 1}', f)] = float(seg[np.argmax(np.abs(seg))])
val = pd.DataFrame([
    ['EOF-period days', WH04['n_days'], best['n_fit']],
    ['EOF1 / EOF2 / EOF3 explained %', '12.8 / 12.2 / 6.1', ' / '.join(f'{100 * v:.1f}' for v in best['frac'][:3])],
    ['30-80 d variance fraction PC1/PC2/PC3', '0.60 / 0.62 / 0.31',
     ' / '.join(f'{band_frac(p, 30, 80):.2f}' for p in (P1, P2, P3))],
    ['mean Coh^2(PC1,PC2), 30-80 d (Welch, approx.)', 0.76, round(coh2_band(P1, P2, 30, 80), 2)],
    ['max lag corr RMM1->RMM2 (r, lag d)', '0.56, 9', '%.2f, %d' % lagcorr_max(P1, P2)],
    ['EOF1 largest |value| OLR/u850/u200 (signed)', '-8.5 / 1.2 / 2.7', ' / '.join(f"{ext[('EOF1', f)]:.1f}" for f in FIELDS)],
    ['EOF2 largest |value| OLR/u850/u200 (signed)', '11.7 / 1.5 / 2.8', ' / '.join(f"{ext[('EOF2', f)]:.1f}" for f in FIELDS)],
    ['normalisation factors OLR/u850/u200', '(not printed in WH04)', ' / '.join(f"{best['fac'][f]:.2f}" for f in FIELDS)],
], columns=['quantity', 'WH04', 'reproduction'])
print('VS THE PAPER')
print(val.to_string(index=False))

cmp_rows = [{'period': p, **compare(best['pcn'], m)} for p, m in PERIODS.items()]
cmp = pd.DataFrame(cmp_rows)
print('\nVS THE OFFICIAL BoM RMM  (r = same component; r12/r21 = crossed; canon = rotation-free plane match)')
print(cmp.round(3).to_string(index=False))
flips = [int(np.sign(cmp.loc[0, 'r1'])), int(np.sign(cmp.loc[0, 'r2']))]
print(f'\nSign vs BoM (WH04 Fig. 1 convention): RMM1 {flips[0]:+d}, RMM2 {flips[1]:+d}  (+1 = same sign)')

## Cell 6 — Ladder row R3: the same WH04 method on the ERA5 fields

The input is `X_MJO.npy` from nb13: ERA5 daily means, 15S-15N mean, 3-harmonic annual cycle over 1979-2001,
previous-120-day mean removed. That is the WH04 recipe without SST1, but on a 2° grid. Here it is interpolated
to the 2.5° longitudes, each field is re-normalised by its global standard deviation, and the EOF is fitted on
1979-2001. Ladder row R4 (nb24's own-RMM) differs from this only in its EOF period (1979-2023) and its 2° grid.

In [ ]:
X = np.load(f'{MJO_PROC}/X_MJO.npy', mmap_mode='r')                  # (N, 3, 1, 180): [u850, OLR, u200]
lon_e = np.load(f'{MJO_PROC}/longitudes_mjo.npy').astype(float)
d_e = pd.DatetimeIndex(lab.index)
assert len(X) == len(d_e), 'X_MJO rows vs labels_aligned_mjo rows'
lon25 = np.arange(0, 360, 2.5)
pos = np.interp(lon25, np.r_[lon_e, lon_e[0] + 360], np.arange(len(lon_e) + 1))   # periodic interpolation weights
i0 = np.floor(pos).astype(int); wgt = pos - i0; i1 = (i0 + 1) % len(lon_e); i0 = i0 % len(lon_e)
base_e = (d_e >= EOF_PERIOD[0]) & (d_e <= EOF_PERIOD[1])
fac_e, parts = {}, []
for f, ch in zip(FIELDS, [1, 0, 2]):
    F = np.asarray(X[:, ch, 0, :], dtype=np.float64)
    F = F[:, i0] * (1 - wgt) + F[:, i1] * wgt
    fac_e[f] = global_sd(F, base_e)
    parts.append(F / fac_e[f])
era5 = finish(np.concatenate(parts, axis=1), fac_e, base_e, lon25, {'source': 'ERA5 X_MJO (nb13), 2.5 deg'})
off_e = lab[['rmm1', 'rmm2']].values.astype(float); off_e[np.abs(off_e) > 100] = np.nan
m = base_e & np.isfinite(era5['pcn'][:, :2]).all(1) & np.isfinite(off_e).all(1)
r_e = [np.corrcoef(era5['pcn'][m, k], off_e[m, k])[0, 1] for k in range(2)]
c_e = canon(era5['pcn'][m, :2], off_e[m])
print(f"R3 ERA5: EOF-period days {era5['n_fit']}, explained {100 * era5['frac'][0]:.1f} / {100 * era5['frac'][1]:.1f} %, "
      f'r vs BoM (1979-2001) {r_e[0]:+.3f} / {r_e[1]:+.3f}, canonical {c_e[0]:.3f} / {c_e[1]:.3f}')

## Cell 7 — Figures: EOF structures (cf. WH04 Fig. 1) and reproduction vs official

In [ ]:
INK, INK2, GRID, BASE, SURF = '#0b0b0b', '#52514e', '#e1e0d9', '#c3c2b7', '#fcfcfb'
STYLE = {'olr': ('-', '#1c5cab', 'OLR'), 'u850': ('--', '#eb6834', 'u850'), 'u200': (':', '#1baf7a', 'u200')}

def tidy(ax):
    ax.set_facecolor(SURF); ax.grid(color=GRID, lw=0.8); ax.set_axisbelow(True)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['left', 'bottom']: ax.spines[s].set_color(BASE)
    ax.tick_params(colors=INK2, labelsize=9)

fig, axes = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True, facecolor=SURF)
for k, ax in enumerate(axes):
    ph = best['E'][k] * np.sqrt(best['lam'][k])
    for i, f in enumerate(FIELDS):
        ls, c, name = STYLE[f]
        seg = ph[i * len(lon):(i + 1) * len(lon)]
        j = np.argmax(np.abs(seg))
        ax.plot(lon, seg, ls=ls, color=c, lw=2, label=f'{name} (extreme {seg[j] * best["fac"][f]:+.1f} '
                f'{"W m-2" if f == "olr" else "m s-1"})')
    ax.axhline(0, color=BASE, lw=1); tidy(ax)
    ax.set_ylabel('normalised magnitude', color=INK2)
    ax.set_title(f'EOF{k + 1}  ({100 * best["frac"][k]:.1f}% of variance; WH04: {WH04["expl_var_pct"][k]}%)',
                 loc='left', fontsize=11, color=INK)
    ax.legend(frameon=False, fontsize=8, labelcolor=INK2, loc='lower left')
axes[1].set_xlabel('longitude (deg E)', color=INK2); axes[1].set_xlim(0, 360)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/rmm_repro_eofs.png', dpi=150, bbox_inches='tight', facecolor=SURF); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), facecolor=SURF, gridspec_kw={'width_ratios': [1, 1, 2]})
m = PERIODS[p0] & np.isfinite(best['pcn'][:, :2]).all(1) & np.isfinite(off).all(1)
for k, ax in enumerate(axes[:2]):
    sgn = np.sign(cmp.loc[0, f'r{k + 1}'])
    ax.hexbin(off[m, k], sgn * best['pcn'][m, k], gridsize=45, cmap='Blues', mincnt=1)
    ax.plot([-4, 4], [-4, 4], color=INK2, lw=1, ls='--'); tidy(ax)
    ax.set_xlabel(f'official RMM{k + 1}', color=INK2); ax.set_ylabel(f'reproduced RMM{k + 1}', color=INK2)
    ax.set_title(f'RMM{k + 1}, 1979-2001: r = {cmp.loc[0, f"r{k + 1}"]:+.3f}', loc='left', fontsize=10, color=INK)
ax = axes[2]; yr = (dates >= '2001-01-01') & (dates <= '2001-12-31')
for k, c in enumerate(['#1c5cab', '#eb6834']):
    sgn = np.sign(cmp.loc[0, f'r{k + 1}'])
    ax.plot(dates[yr], off[yr, k], color=c, lw=1.2, label=f'official RMM{k + 1}')
    ax.plot(dates[yr], sgn * best['pcn'][yr, k], color=c, lw=1.2, ls='--', label=f'reproduced RMM{k + 1}')
tidy(ax); ax.set_title('2001 (WH04 Fig. 5: official = normalised PCs in 2001)', loc='left', fontsize=10, color=INK)
ax.legend(frameon=False, fontsize=8, ncol=2, labelcolor=INK2)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/rmm_repro_vs_official.png', dpi=150, bbox_inches='tight', facecolor=SURF)
plt.show()

## Cell 8 — Save the reproduced indices (read by nb35 Cell 12)

In [ ]:
def save_index(path, run, d, source, extra):
    pcn = run['pcn']
    ok = np.isfinite(pcn[:, :2]).all(1)
    ph = np.zeros(len(pcn), int)
    ph[ok] = wh04_phase(pcn[ok, 0], pcn[ok, 1])
    meta = {'source': source, 'method': 'WH04 without SST1 regression', 'eof_period': EOF_PERIOD,
            'expl_var_pct': [round(100 * v, 2) for v in run['frac'][:3]], 'n_fit_days': run['n_fit'],
            'normalisation': {f: round(v, 4) for f, v in run['fac'].items()}, 'config': run['cfg'], **extra}
    np.savez_compressed(path, dates=np.asarray(d.strftime('%Y-%m-%d')), pcs=pcn[:, :2].astype(np.float32),
                        pcs_raw=run['pcs'].astype(np.float32), eofs=run['E'][:3].astype(np.float32),
                        eigvals=run['lam'], phase=ph, amp=np.hypot(pcn[:, 0], pcn[:, 1]), meta=json.dumps(meta, default=float))
    print('saved', path)

save_index(f'{IDX_DIR}/rmm_wh04_repro.npz', best, dates, 'NOAA interpolated OLR + NCEP-NCAR R1 (2.5 deg)',
           {'vs_official': cmp.round(4).to_dict(orient='records')})
save_index(f'{IDX_DIR}/rmm_wh04_era5.npz', era5, d_e, 'ERA5 daily means (X_MJO, nb13), regridded to 2.5 deg',
           {'r_vs_official_1979_2001': [round(r_e[0], 4), round(r_e[1], 4)],
            'canon_vs_official_1979_2001': [round(float(c), 4) for c in c_e]})
json.dump({'best_variant': BEST, 'paper_vs_repro': val.astype(str).to_dict(orient='records'),
           'vs_official': cmp.round(4).to_dict(orient='records'), 'sst1_used': USE_SST1},
          open(f'{OUT_DIR}/rmm_validation.json', 'w'), indent=2, default=float)
print('saved', f'{OUT_DIR}/rmm_validation.json')

---
## Done — what to send back

- The Cell 4 grid table.
- The two Cell 5 tables (vs the paper, vs BoM).
- The Cell 6 line (R3).
- `rmm_repro_eofs.png` and `rmm_repro_vs_official.png`.

**How to read it:**
- **Per-component r with BoM >= 0.99 in 1979-2001:** the reproduction is exact apart from the SST1 term.
- **Lower r in 1979-2013 than in 2014-2022:** that is the SST1 step, which only the earlier official values contain.
- **Canonical correlations near 1:** the 2-D plane is the same. For the ENSO displacement test, which whitens the plane, this is what matters.